<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Session 4 — Probability, Statistics, and Regression for Social Science</h2>
</div>

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Learning Goals</h2>
<ul>
<li>Understand why probability matters for data analysis</li>
<li>Connect descriptive statistics to uncertainty and sampling</li>
<li>Interpret correlation and regression in social science settings</li>
<li>Fit and explain simple and multiple linear regression models in Python</li>
<li>Distinguish prediction from causal claims</li>
</ul>
</div>

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Why regression in social science?</h2>
Social scientists often ask questions like: Do students who attend class more score higher? Is stress associated with lower well-being? Does parental education predict college persistence? Regression is one of the main tools used to answer questions like these because it helps us describe relationships between variables.<br><br>
But regression does not stand alone. To understand it well, we need several prerequisites: distributions, probability, sampling, variation, and correlation. This notebook builds those pieces step by step.
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
np.random.seed(42)

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Our example dataset</h2>
To keep the notebook self-contained, we will generate a synthetic dataset about undergraduate students. The variables are designed to resemble a social science study about academic performance and student life. Because the data are simulated, we know the general structure of the relationships, which makes them useful for teaching.
</div>

In [ ]:
n = 250

attendance = np.random.normal(82, 10, n).clip(45, 100)
study_hours = np.random.normal(11, 3, n).clip(2, 22)
stress = np.random.normal(5.5, 1.8, n).clip(1, 10)
sleep_hours = np.random.normal(6.8, 1.0, n).clip(4, 9)
parent_education = np.random.choice([12, 14, 16, 18], size=n, p=[0.18, 0.27, 0.37, 0.18])
community_engagement = np.random.normal(4, 2.5, n).clip(0, 12)

noise = np.random.normal(0, 6, n)
exam_score = (
    18
    + 0.42 * attendance
    + 1.7 * study_hours
    - 1.4 * stress
    + 1.1 * sleep_hours
    + 0.35 * parent_education
    + noise
).clip(40, 100)

df = pd.DataFrame({
    "attendance": attendance,
    "study_hours": study_hours,
    "stress": stress,
    "sleep_hours": sleep_hours,
    "parent_education": parent_education,
    "community_engagement": community_engagement,
    "exam_score": exam_score
})

df["passed"] = np.where(df["exam_score"] >= 70, "Yes", "No")
df.head()

## 1. Descriptive Statistics: The First Step

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Why start with descriptive statistics?</h2>
Before we model anything, we need to understand what the data look like. Descriptive statistics summarize the center and spread of a variable.
<ul>
<li><b>Mean</b>: the average value</li>
<li><b>Median</b>: the middle value</li>
<li><b>Standard deviation</b>: how spread out the values are</li>
</ul>
In social science, this helps us answer basic questions such as: What does a typical student look like? How much variation is there? Are some students very different from others?
</div>

In [ ]:
df.describe().round(2)

### Example interpretation

If the average exam score is around the mid-to-high 70s, that tells us the class is doing reasonably well overall. But the standard deviation matters too: a large spread would mean some students are doing much better or much worse than average.

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Compute the mean, median, and standard deviation for <code>stress</code> and <code>sleep_hours</code>. Then write one sentence describing what those numbers suggest about the students in this dataset.
</div>

In [ ]:
# Your code here


## 2. Probability Basics

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">What is probability?</h2>
Probability describes uncertainty. When we say an event has probability 0.70, we mean that in the long run we expect it to happen about 70% of the time.<br><br>
In social science, probability helps us think about uncertainty in outcomes. For example:
<ul>
<li>What is the probability that a student passes?</li>
<li>What is the probability of high stress given low sleep?</li>
<li>If we draw a random sample, how likely is it that our sample average differs from the population average?</li>
</ul>
</div>

In [ ]:
pass_prob = (df["passed"] == "Yes").mean()
print(f"Estimated probability of passing: {pass_prob:.3f}")

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Conditional probability</h2>
A conditional probability asks: what is the probability of one event <i>given</i> that another event has happened?<br><br>
For instance: among students with high attendance, what proportion pass? That question is more informative than the overall pass rate because it focuses on a subgroup.
</div>

In [ ]:
high_attendance = df["attendance"] >= 85
cond_prob = (df.loc[high_attendance, "passed"] == "Yes").mean()

print(f"P(pass | attendance >= 85) = {cond_prob:.3f}")

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Calculate the probability that a student passes given that their stress score is below 5. Then compare it to the overall pass probability.
</div>

In [ ]:
# Your code here


## 3. Distributions and Variation

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Why distributions matter</h2>
Regression tries to summarize relationships between variables. To do that responsibly, we should first inspect the distributions of those variables. A distribution tells us how values are spread across low, medium, and high ranges.<br><br>
In social science, distributions help us notice patterns such as:
<ul>
<li>Are most people near the center?</li>
<li>Are there outliers?</li>
<li>Is the variable skewed?</li>
</ul>
</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["exam_score"], bins=18, kde=True, ax=axes[0], color="#4f46e5")
axes[0].set_title("Distribution of Exam Scores")

sns.histplot(df["stress"], bins=14, kde=True, ax=axes[1], color="#ef4444")
axes[1].set_title("Distribution of Stress Scores")

plt.tight_layout()
plt.show()

### Example interpretation

If exam scores cluster near the center with a roughly bell-shaped distribution, then the class performance is fairly balanced. If stress is skewed, then a smaller number of students may be experiencing especially high stress.

## 4. Sampling and Sampling Variability

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Why sampling matters for regression</h2>
We usually do not observe an entire population. Instead, we observe a sample. This means our statistics and regression coefficients are not fixed truths. They are estimates that would change a little if we collected a different sample.<br><br>
This is one reason probability is so important in statistics: it helps us understand how much random variation to expect from sample to sample.
</div>

In [ ]:
sample_means = []

for _ in range(500):
    sample = df["exam_score"].sample(30, replace=True)
    sample_means.append(sample.mean())

plt.figure(figsize=(8, 4))
sns.histplot(sample_means, bins=20, color="#0f766e")
plt.axvline(df["exam_score"].mean(), color="black", linestyle="--", label="Population mean")
plt.title("Sampling Distribution of the Mean Exam Score")
plt.legend()
plt.show()

### What is this plot showing?

Each bar summarizes the mean of a different random sample of 30 students. The means vary, but they cluster around the overall mean. This is the core idea behind sampling variability.

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Repeat the sampling exercise with samples of size 10 instead of 30. Does the sampling distribution become more spread out or less spread out?
</div>

In [ ]:
# Your code here


## 5. Covariance and Correlation

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Correlation is a prerequisite for regression</h2>
Regression becomes useful when variables move together in systematic ways. Correlation summarizes the strength and direction of a linear relationship between two numeric variables.<br><br>
The correlation coefficient ranges from -1 to 1:
<ul>
<li>Positive values: as one variable increases, the other tends to increase</li>
<li>Negative values: as one variable increases, the other tends to decrease</li>
<li>Values near 0: weak linear relationship</li>
</ul>
In social science, correlation might help us describe patterns like stress and sleep, or attendance and grades. But correlation alone does not explain why the relationship exists.
</div>

In [ ]:
corr = df[["attendance", "study_hours", "stress", "sleep_hours", "exam_score"]].corr().round(2)
corr

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.show()

### Social science connection

Suppose a researcher finds that higher attendance is strongly correlated with higher exam scores. That is meaningful, but it still does not prove attendance causes achievement. Students who attend more may also differ in motivation, preparation, or outside support.

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Find the correlation between <code>stress</code> and <code>exam_score</code>. Is it positive or negative? What might that mean in a real student population?
</div>

In [ ]:
# Your code here


## 6. Regression Intuition

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">What does regression do?</h2>
Regression fits a line that summarizes the relationship between a predictor variable and an outcome variable. In simple linear regression, the line has the form:<br><br>
<code>y = b0 + b1x</code><br><br>
where:
<ul>
<li><code>b0</code> is the intercept</li>
<li><code>b1</code> is the slope</li>
</ul>
The slope tells us how much we expect <code>y</code> to change when <code>x</code> increases by one unit, on average. In a social science context, that might mean how much exam score changes for each additional hour studied.
</div>

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=df, x="study_hours", y="exam_score", scatter_kws={"alpha": 0.65})
plt.title("Study Hours and Exam Score")
plt.show()

### Reading the plot

Each point is a student. The fitted line summarizes the average trend. If the line slopes upward, then students who study more tend to score higher.

## 7. Simple Linear Regression in Python

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Fitting a regression model</h2>
Now we will fit a simple linear regression model where <code>study_hours</code> predicts <code>exam_score</code>. This gives us a numerical estimate of the slope and intercept, not just a visual impression.
</div>

In [ ]:
X = df[["study_hours"]]
y = df["exam_score"]

model = LinearRegression()
model.fit(X, y)

pred = model.predict(X)

print("Intercept:", round(model.intercept_, 3))
print("Slope for study_hours:", round(model.coef_[0], 3))
print("R-squared:", round(r2_score(y, pred), 3))

### How do we interpret the slope?

If the slope is about 1.8, then for each additional hour of study, the predicted exam score increases by about 1.8 points on average. That is a substantive interpretation, not just a mathematical one.

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">What is R-squared?</h2>
R-squared tells us how much of the variation in the outcome is explained by the model. If R-squared is 0.30, that means 30% of the variation in exam scores is explained by study hours alone.<br><br>
In social science, even modest R-squared values can still be meaningful, because human behavior is shaped by many factors at once.
</div>

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Fit a simple linear regression model where <code>attendance</code> predicts <code>exam_score</code>. Then compare its slope and R-squared to the model using <code>study_hours</code>.
</div>

In [ ]:
# Your code here


## 8. Residuals and Model Error

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">What is a residual?</h2>
A residual is the difference between an observed value and the value predicted by the regression line.<br><br>
<code>residual = observed - predicted</code><br><br>
Residuals matter because they tell us how much error remains after we fit the model. A regression line is not trying to pass through every point. It is trying to minimize overall prediction error.
</div>

In [ ]:
residuals = y - pred

plt.figure(figsize=(8, 4))
sns.scatterplot(x=pred, y=residuals, alpha=0.65)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Predicted exam score")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.show()

print("RMSE:", round(np.sqrt(mean_squared_error(y, pred)), 3))

## 9. Multiple Regression

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Why move beyond one predictor?</h2>
Social science outcomes usually depend on more than one thing. Exam scores may reflect study hours, attendance, stress, sleep, prior preparation, and family resources. Multiple regression lets us examine several predictors at the same time.<br><br>
This matters because a variable may look important in a simple model but become weaker after we account for other relevant factors.
</div>

In [ ]:
X_multi = df[["study_hours", "attendance", "stress", "sleep_hours", "parent_education"]]
y = df["exam_score"]

multi_model = LinearRegression()
multi_model.fit(X_multi, y)
multi_pred = multi_model.predict(X_multi)

coef_table = pd.DataFrame({
    "predictor": X_multi.columns,
    "coefficient": multi_model.coef_
}).round(3)

print("Intercept:", round(multi_model.intercept_, 3))
print("R-squared:", round(r2_score(y, multi_pred), 3))
coef_table

### How do we interpret a multiple regression coefficient?

If the coefficient for `study_hours` is 1.5 in the multiple regression, that means: holding attendance, stress, sleep, and parent education constant, one additional hour of study is associated with a 1.5-point increase in exam score on average.

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Fit a multiple regression model that predicts <code>exam_score</code> using <code>attendance</code>, <code>stress</code>, and <code>sleep_hours</code>. Which predictor seems strongest based on coefficient size?
</div>

In [ ]:
# Your code here


## 10. Regression Does Not Automatically Mean Causation

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Prediction vs. explanation vs. causation</h2>
Regression is excellent for describing associations and making predictions. But a regression coefficient does not automatically tell us that one variable causes another.<br><br>
For causal claims, social scientists also need research design: experiments, strong theory, longitudinal data, quasi-experiments, or careful control of confounding variables.<br><br>
So if attendance predicts exam score, we should say attendance is <i>associated with</i> exam performance unless we have a stronger design that supports a causal interpretation.
</div>

## 11. Mini Case Study

Imagine a social science researcher asks:

> Are students with higher community engagement also performing better academically?

This is a meaningful question because community engagement may reflect motivation, social support, or time management. But it could also compete with study time. We can explore the relationship statistically.

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=df, x="community_engagement", y="exam_score", scatter_kws={"alpha": 0.65}, color="#7c3aed")
plt.title("Community Engagement and Exam Score")
plt.show()

print("Correlation:", round(df["community_engagement"].corr(df["exam_score"]), 3))

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Write 2 or 3 sentences interpreting the relationship between community engagement and exam score. Then explain one reason why this relationship should not automatically be treated as causal.
</div>

In [ ]:
# Write your interpretation here as comments or print statements


## 12. Summary

In this notebook, you learned that regression depends on several foundational ideas:

- Descriptive statistics summarize the center and spread of variables.
- Probability helps us think about uncertainty.
- Sampling variability explains why sample-based estimates move around.
- Correlation tells us whether variables move together linearly.
- Regression fits a line or model that summarizes how predictors relate to an outcome.
- Multiple regression lets us include several predictors at once.
- Regression is powerful, but association is not the same as causation.

These tools are used constantly in social science research to answer questions about education, inequality, health, behavior, and social life.

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Final Practice</h2>
Choose one social science question that this dataset could help answer. Then identify:
<ol>
<li>the outcome variable,</li>
<li>one or more predictors, and</li>
<li>what kind of regression or statistical summary you would use.</li>
</ol>
Write a short explanation of why your analysis would be useful.
</div>

In [ ]:
# Your response here
